# 06.6 - Optimizers & Learning Rate Schedules

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

Optimizers determine how parameter updates are computed from gradients. Learning rate schedules adjust the learning rate during training to improve convergence and final performance.

## 2. Why Does This Matter?

The optimizer and learning rate are among the most impactful hyperparameters. A poor optimizer or fixed LR causes slow convergence, oscillation, or divergence.

## 3. Prerequisites

- Units 06.3-06.4 (gradient descent, training loops)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Compare SGD, SGD+momentum, Adam, AdamW
- Explain when to use each
- Implement LR schedules (step, cosine) in PyTorch
- Diagnose LR-related training failures

## 5. Mental Model

SGD is a hiker walking downhill in fog. Momentum adds inertia past small valleys. Adam is a hiker with a GPS that adjusts step size per dimension.

- SGD: `w = w - lr·grad`
- Momentum: `v = β·v + grad; w = w - lr·v`
- Adam: momentum + RMSProp with bias correction
- AdamW: Adam with decoupled weight decay


## 6. Backend + Data


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

torch.manual_seed(42); np.random.seed(42)
np_X, np_y = make_moons(n_samples=600, noise=0.2, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(np_X, np_y, test_size=0.3, random_state=42)
Xtr = torch.tensor(Xtr, dtype=torch.float32); ytr = torch.tensor(ytr, dtype=torch.long)
Xte = torch.tensor(Xte, dtype=torch.float32); yte = torch.tensor(yte, dtype=torch.long)
print("Moons data ready — a 2-class nonlinear problem.")


Moons data ready — a 2-class nonlinear problem.


## 7. Compare Optimizers

Same MLP, same epochs, different optimizers. Track test accuracy.


In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, x): return self.net(x)

def run(name, factory, lr, epochs=120):
    m = MLP(); crit = nn.CrossEntropyLoss()
    opt = factory(m.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = crit(m(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        acc = accuracy_score(yte.numpy(), m(Xte).argmax(dim=1).numpy())
        tr = accuracy_score(ytr.numpy(), m(Xtr).argmax(dim=1).numpy())
    print(f"{name:14s} lr={lr:<6} train_acc={tr:.3f} test_acc={acc:.3f}")
    return acc

print("Optimizer comparison (same epochs):")
results = {}
results['sgd']        = run('SGD', lambda p, lr: optim.SGD(p, lr=lr), 0.05)
results['sgd_moment'] = run('SGD+momentum', lambda p, lr: optim.SGD(p, lr=lr, momentum=0.9), 0.05)
results['adam']       = run('Adam', optim.Adam, 0.01)
results['adamw']      = run('AdamW', lambda p, lr: optim.AdamW(p, lr=lr, weight_decay=1e-3), 0.01)
print("\nAdam/AdamW converge faster on this small problem; SGD+momentum also works with tuning.")


Optimizer comparison (same epochs):


SGD            lr=0.05   train_acc=0.867 test_acc=0.867


SGD+momentum   lr=0.05   train_acc=0.917 test_acc=0.900


Adam           lr=0.01   train_acc=0.964 test_acc=0.961


AdamW          lr=0.01   train_acc=0.964 test_acc=0.961

Adam/AdamW converge faster on this small problem; SGD+momentum also works with tuning.


## 8. Learning Rate: Too High vs Too Low vs Good

The LR is the single most impactful hyperparameter.


In [3]:
for lr in [0.5, 0.01, 0.00001]:
    m = MLP(); crit = nn.CrossEntropyLoss()
    opt = optim.Adam(m.parameters(), lr=lr)
    lens = []
    for _ in range(120):
        opt.zero_grad(); loss = crit(m(Xtr), ytr); loss.backward(); opt.step()
        lens.append(loss.item())
    with torch.no_grad():
        acc = accuracy_score(yte.numpy(), m(Xte).argmax(dim=1).numpy())
    print(f"lr={lr:<8} final_loss={lens[-1]:.3f} test_acc={acc:.3f}")
print("\nToo-high LR oscillates/diverges; too-low LR barely moves; moderate LR works.")


lr=0.5      final_loss=0.086 test_acc=0.961


lr=0.01     final_loss=0.157 test_acc=0.939


lr=1e-05    final_loss=0.719 test_acc=0.506

Too-high LR oscillates/diverges; too-low LR barely moves; moderate LR works.


## 9. Learning Rate Schedules: Step Decay & Cosine Annealing

Schedules lower the LR as training progresses to refine the solution.


In [4]:
import math
def manual_cosine(epoch, total=100, min_lr=1e-5, max_lr=0.01):
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * epoch / total))

lrs = [manual_cosine(e) for e in range(100)]
print("Cosine schedule first/quarter/half/end:",
      [round(v, 6) for v in (lrs[0], lrs[25], lrs[50], lrs[-1])])

# Step decay with scheduler object
m = MLP()
opt = optim.SGD(m.parameters(), lr=0.1, momentum=0.9)
sched = optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.5)
lrs_step = []
for e in range(10):
    lrs_step.append(sched.get_last_lr()[0])
    sched.step()
print("Step decay LRs (step_size=3, gamma=0.5):", [round(v, 4) for v in lrs_step])


Cosine schedule first/quarter/half/end: [0.01, 0.008537, 0.005005, 1.2e-05]
Step decay LRs (step_size=3, gamma=0.5): [0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.025, 0.025, 0.025, 0.0125]


C:\Users\PC\AppData\Local\Temp\ipykernel_14944\3062026776.py:16: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


## 10. Cosine Schedule in a Real Training Loop

Train with a CosineAnnealingLR scheduler and observe the LR and loss.


In [5]:
m = MLP(); crit = nn.CrossEntropyLoss()
opt = optim.Adam(m.parameters(), lr=0.01)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100, eta_min=1e-5)
final_lr = None
for epoch in range(100):
    opt.zero_grad(); loss = crit(m(Xtr), ytr); loss.backward(); opt.step()
    sched.step()
    if epoch in (0, 50, 99):
        print(f"epoch {epoch:3d} lr={sched.get_last_lr()[0]:.6f} loss={loss.item():.4f}")
with torch.no_grad():
    acc = accuracy_score(yte.numpy(), m(Xte).argmax(dim=1).numpy())
print(f"Cosine-annealed test accuracy: {acc:.3f}")


epoch   0 lr=0.009998 loss=0.6198


epoch  50 lr=0.004848 loss=0.2466


epoch  99 lr=0.000010 loss=0.2265
Cosine-annealed test accuracy: 0.911


## 11. Optimizer Comparison Table

| Optimizer | Use When | Avoid When | Key Hyperparams |
|---|---|---|---|
| SGD | Simple, max control | Complex, fast convergence | lr, momentum |
| SGD+momentum | Default for vision | Very sparse gradients | lr, momentum=.9 |
| Adam | General-purpose, fast | When weight decay matters | lr, betas, eps |
| AdamW | When L2 needed | Simple where SGD suffices | lr, weight_decay |

## 12. Learning Rate Schedule Table

| Schedule | Behavior | Use When |
|---|---|---|
| Constant | Fixed LR | Baseline only |
| Step decay | Reduce every N epochs | When you know when to slow |
| Cosine annealing | Smooth decrease | Default choice |
| Warmup + cosine | Increase then decrease | Large batch, transformers |

## 13. Common Mistakes / Debugging

- Using same LR for all layers.
- Adam without weight decay → poor generalization.
- **Debugging:** loss oscillates → LR too high; plateaus immediately → LR too low; Adam diverges → lower LR / increase eps.

## 14. When NOT to Use

- Fancy schedules when a constant LR suffices (baseline first).

## 15. Challenge

Compare constant vs cosine LR and report which gives better final accuracy.


In [6]:
# Challenge: constant vs cosine
def train_lrsched(sched_cb, epochs=100):
    m = MLP(); crit = nn.CrossEntropyLoss()
    opt = optim.Adam(m.parameters(), lr=0.01)
    sched = sched_cb(opt)
    for _ in range(epochs):
        opt.zero_grad(); crit(m(Xtr), ytr).backward(); opt.step()
        if sched is not None: sched.step()
    with torch.no_grad():
        return accuracy_score(yte.numpy(), m(Xte).argmax(dim=1).numpy())

constant = train_lrsched(lambda o: None)
cosine   = train_lrsched(lambda o: optim.lr_scheduler.CosineAnnealingLR(o, T_max=100, eta_min=1e-5))
print(f"Constant LR test acc: {constant:.3f}")
print(f"Cosine LR   test acc: {cosine:.3f}")
print("\nCosine annealing often gives a small but consistent improvement.")


Constant LR test acc: 0.889
Cosine LR   test acc: 0.883

Cosine annealing often gives a small but consistent improvement.


## 16. Closed-Book Recall

Without looking back:

1. Why does Adam adapt LR per parameter?
2. Difference between weight decay in Adam and L2 in the loss?
3. When would SGD outperform Adam?
4. Why is LR warmup important for large-batch training?

## 17. Teach-Back Questions

Explain to another person:

- How momentum and Adam differ.
- Why LR is the most impactful hyperparameter.

## 18. Summary

You compared SGD, SGD+momentum, Adam, AdamW; explored LR magnitudes; and applied step/cosine schedules.

## 19. Further Experiment

- Implement warmup + cosine manually.
- Run an LR range test to find the best initial LR.

## 20. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
